In [1]:
import os
import time
import json
from typing import Dict, Tuple

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms



class MNISTMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28 * 28, 128)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(128, 64)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(64, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1) 
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)           
        return x


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = running_loss / total
    acc = correct / total
    return avg_loss, acc


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        loss = criterion(logits, labels)

        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = running_loss / total
    acc = correct / total
    return avg_loss, acc


def symmetric_quantize_tensor(t: torch.Tensor) -> Tuple[torch.Tensor, float]:
    """
    Quantize float tensor to int8 using symmetric quantization.
    q = round(t / scale), clipped to [-128, 127]
    """
    max_abs = t.abs().max().item()
    scale = max_abs / 127.0 if max_abs > 0 else 1.0
    q = torch.clamp(torch.round(t / scale), -128, 127).to(torch.int8)
    return q, scale


@torch.no_grad()
def collect_activation_ranges(model: nn.Module, loader: DataLoader, device: torch.device, num_batches: int = 20):
    """
    Collect max abs activation ranges for each layer output.
    Used to estimate activation scales for INT8 quantization.
    """
    model.eval()

    stats = {
        "input": 0.0,
        "fc1_out": 0.0,
        "relu1_out": 0.0,
        "fc2_out": 0.0,
        "relu2_out": 0.0,
        "fc3_out": 0.0,
    }

    batch_count = 0
    for images, _ in loader:
        images = images.to(device)
        x = images.view(images.size(0), -1)

        stats["input"] = max(stats["input"], x.abs().max().item())

        fc1_out = model.fc1(x)
        stats["fc1_out"] = max(stats["fc1_out"], fc1_out.abs().max().item())

        relu1_out = model.relu1(fc1_out)
        stats["relu1_out"] = max(stats["relu1_out"], relu1_out.abs().max().item())

        fc2_out = model.fc2(relu1_out)
        stats["fc2_out"] = max(stats["fc2_out"], fc2_out.abs().max().item())

        relu2_out = model.relu2(fc2_out)
        stats["relu2_out"] = max(stats["relu2_out"], relu2_out.abs().max().item())

        fc3_out = model.fc3(relu2_out)
        stats["fc3_out"] = max(stats["fc3_out"], fc3_out.abs().max().item())

        batch_count += 1
        if batch_count >= num_batches:
            break

    scales = {}
    for k, v in stats.items():
        scales[k] = (v / 127.0) if v > 0 else 1.0

    return stats, scales


@torch.no_grad()
def quantize_model_weights(model: nn.Module) -> Dict[str, Dict[str, torch.Tensor]]:
    """
    Quantize all FC weights and biases.
    Biases are stored as float here for simplicity.
    If you want pure integer pipeline later, bias can be quantized to int32.
    """
    qparams = {}

    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            w_q, w_scale = symmetric_quantize_tensor(module.weight.data.cpu())
            # Bias kept as float for now; easier and acceptable for midpoint software side
            b = module.bias.data.cpu() if module.bias is not None else None

            qparams[name] = {
                "weight_int8": w_q,
                "weight_scale": torch.tensor([w_scale], dtype=torch.float32),
                "bias_fp32": b if b is not None else torch.tensor([], dtype=torch.float32),
            }

    return qparams


def save_quantized_artifacts(model: nn.Module,
                             qweights: Dict[str, Dict[str, torch.Tensor]],
                             act_scales: Dict[str, float],
                             out_dir: str = "artifacts"):
    os.makedirs(out_dir, exist_ok=True)

    # Save full float model
    torch.save(model.state_dict(), os.path.join(out_dir, "mnist_mlp_fp32.pth"))

    # Save quantized tensors
    save_dict = {}
    for layer_name, info in qweights.items():
        save_dict[f"{layer_name}.weight_int8"] = info["weight_int8"]
        save_dict[f"{layer_name}.weight_scale"] = info["weight_scale"]
        save_dict[f"{layer_name}.bias_fp32"] = info["bias_fp32"]

    torch.save(save_dict, os.path.join(out_dir, "mnist_mlp_int8_weights.pth"))

    # Save activation scales as json
    with open(os.path.join(out_dir, "activation_scales.json"), "w") as f:
        json.dump(act_scales, f, indent=2)

    print(f"Saved artifacts to: {out_dir}")



@torch.no_grad()
def fake_quantize_activation(x: torch.Tensor, scale: float) -> torch.Tensor:
    q = torch.clamp(torch.round(x / scale), -128, 127)
    dq = q * scale
    return dq


@torch.no_grad()
def evaluate_fake_quantized(model: nn.Module, loader: DataLoader, device: torch.device, act_scales: Dict[str, float]):
    model.eval()
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        x = images.view(images.size(0), -1)
        x = fake_quantize_activation(x, act_scales["input"])

        x = model.fc1(x)
        x = fake_quantize_activation(x, act_scales["fc1_out"])
        x = model.relu1(x)
        x = fake_quantize_activation(x, act_scales["relu1_out"])

        x = model.fc2(x)
        x = fake_quantize_activation(x, act_scales["fc2_out"])
        x = model.relu2(x)
        x = fake_quantize_activation(x, act_scales["relu2_out"])

        x = model.fc3(x)
        x = fake_quantize_activation(x, act_scales["fc3_out"])

        preds = torch.argmax(x, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return correct / total



def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    batch_size = 128
    epochs = 5
    learning_rate = 1e-3

    transform = transforms.Compose([
        transforms.ToTensor(),   # [0,1]
    ])

    train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
    test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    model = MNISTMLP().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    print("\n===== Training FP32 model =====")
    best_test_acc = 0.0

    for epoch in range(epochs):
        start = time.time()

        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        test_loss, test_acc = evaluate(model, test_loader, criterion, device)

        elapsed = time.time() - start
        print(
            f"Epoch [{epoch+1}/{epochs}] | "
            f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f} | "
            f"test_loss={test_loss:.4f}, test_acc={test_acc:.4f} | "
            f"time={elapsed:.2f}s"
        )

        if test_acc > best_test_acc:
            best_test_acc = test_acc

    print(f"\nBest FP32 test accuracy: {best_test_acc:.4f}")

    print("\n===== Collect activation ranges =====")
    act_stats, act_scales = collect_activation_ranges(model, train_loader, device, num_batches=20)
    print("Activation max abs stats:")
    for k, v in act_stats.items():
        print(f"  {k}: {v:.6f}")
    print("Activation scales:")
    for k, v in act_scales.items():
        print(f"  {k}: {v:.8f}")

    print("\n===== Quantize weights to INT8 =====")
    qweights = quantize_model_weights(model)
    for layer_name, info in qweights.items():
        print(f"{layer_name}: weight_scale={info['weight_scale'].item():.8f}, shape={tuple(info['weight_int8'].shape)}")

    print("\n===== Fake quantized evaluation =====")
    fq_acc = evaluate_fake_quantized(model, test_loader, device, act_scales)
    print(f"Fake-quantized test accuracy: {fq_acc:.4f}")

    print("\n===== Save artifacts =====")
    save_quantized_artifacts(model, qweights, act_scales, out_dir="artifacts")


if __name__ == "__main__":
    main()

Using device: cpu


100.0%
100.0%
100.0%
100.0%


===== Training FP32 model =====


Epoch [1/5] | train_loss=0.4162, train_acc=0.8867 | test_loss=0.2072, test_acc=0.9378 | time=3.23s
Epoch [2/5] | train_loss=0.1712, train_acc=0.9500 | test_loss=0.1410, test_acc=0.9595 | time=2.78s
Epoch [3/5] | train_loss=0.1223, train_acc=0.9638 | test_loss=0.1113, test_acc=0.9671 | time=2.76s
Epoch [4/5] | train_loss=0.0940, train_acc=0.9714 | test_loss=0.0967, test_acc=0.9692 | time=2.77s
Epoch [5/5] | train_loss=0.0734, train_acc=0.9779 | test_loss=0.0913, test_acc=0.9696 | time=2.82s

Best FP32 test accuracy: 0.9696

===== Collect activation ranges =====
Activation max abs stats:
  input: 1.000000
  fc1_out: 14.954649
  relu1_out: 9.856448
  fc2_out: 13.251422
  relu2_out: 13.251422
  fc3_out: 23.517551
Activation scales:
  input: 0.00787402
  fc1_out: 0.11775314
  relu1_out: 0.07760983
  fc2_out: 0.10434190
  relu2_out: 0.10434190
  fc3_out: 0.18517757

===== Quantize weights to INT8 =====
fc1: weight_scale=0.00461925, shape=(128, 784)
fc2: weight_scale=0.00467464, shape=(64, 12

In [3]:
import json
import time
from typing import Dict, Tuple

import numpy as np
import torch
from torchvision import datasets, transforms



def quantize_to_int8(x_fp32: np.ndarray, scale: float) -> np.ndarray:
    """
    Symmetric quantization:
        q = round(x / scale), clipped to [-128, 127]
    """
    q = np.round(x_fp32 / scale)
    q = np.clip(q, -128, 127).astype(np.int8)
    return q


def dequantize_from_int(x_int: np.ndarray, scale: float) -> np.ndarray:
    """
    Dequantize integer tensor back to float:
        x = q * scale
    """
    return x_int.astype(np.float32) * scale



def gemm_int8(A_int8: np.ndarray, B_int8: np.ndarray) -> np.ndarray:

    A_int32 = A_int8.astype(np.int32)
    B_int32 = B_int8.astype(np.int32)
    C_int32 = A_int32 @ B_int32
    return C_int32


def linear_int8(
    x_int8: np.ndarray,
    w_int8: np.ndarray,
    x_scale: float,
    w_scale: float,
    bias_fp32: np.ndarray = None
) -> np.ndarray:

    y_int32 = gemm_int8(x_int8, w_int8.T)
    y_fp32 = y_int32.astype(np.float32) * (x_scale * w_scale)

    if bias_fp32 is not None and bias_fp32.size > 0:
        y_fp32 += bias_fp32.reshape(1, -1)

    return y_fp32



def relu_fp32(x: np.ndarray) -> np.ndarray:
    return np.maximum(x, 0.0).astype(np.float32)


def softmax_fp32(logits: np.ndarray) -> np.ndarray:
    logits = logits - np.max(logits, axis=1, keepdims=True)
    exp_x = np.exp(logits)
    probs = exp_x / np.sum(exp_x, axis=1, keepdims=True)
    return probs.astype(np.float32)



def load_artifacts(
    weight_path: str = "artifacts/mnist_mlp_int8_weights.pth",
    scale_path: str = "artifacts/activation_scales.json"
) -> Tuple[Dict, Dict]:
    qdict = torch.load(weight_path, map_location="cpu")
    with open(scale_path, "r") as f:
        act_scales = json.load(f)

    params = {
        "fc1": {
            "weight_int8": qdict["fc1.weight_int8"].numpy(),
            "weight_scale": float(qdict["fc1.weight_scale"].item()),
            "bias_fp32": qdict["fc1.bias_fp32"].numpy(),
        },
        "fc2": {
            "weight_int8": qdict["fc2.weight_int8"].numpy(),
            "weight_scale": float(qdict["fc2.weight_scale"].item()),
            "bias_fp32": qdict["fc2.bias_fp32"].numpy(),
        },
        "fc3": {
            "weight_int8": qdict["fc3.weight_int8"].numpy(),
            "weight_scale": float(qdict["fc3.weight_scale"].item()),
            "bias_fp32": qdict["fc3.bias_fp32"].numpy(),
        },
    }

    return params, act_scales


def predict_one_manual(
    image_fp32: np.ndarray,
    params: Dict,
    act_scales: Dict
) -> Tuple[int, np.ndarray, np.ndarray]:
    x0 = image_fp32.reshape(1, -1).astype(np.float32)
    x0_int8 = quantize_to_int8(x0, act_scales["input"])

    # FC1
    fc1_out_fp32 = linear_int8(
        x_int8=x0_int8,
        w_int8=params["fc1"]["weight_int8"],
        x_scale=act_scales["input"],
        w_scale=params["fc1"]["weight_scale"],
        bias_fp32=params["fc1"]["bias_fp32"],
    )
    fc1_relu_fp32 = relu_fp32(fc1_out_fp32)

    fc1_relu_int8 = quantize_to_int8(fc1_relu_fp32, act_scales["relu1_out"])

    # FC2
    fc2_out_fp32 = linear_int8(
        x_int8=fc1_relu_int8,
        w_int8=params["fc2"]["weight_int8"],
        x_scale=act_scales["relu1_out"],
        w_scale=params["fc2"]["weight_scale"],
        bias_fp32=params["fc2"]["bias_fp32"],
    )
    fc2_relu_fp32 = relu_fp32(fc2_out_fp32)

    fc2_relu_int8 = quantize_to_int8(fc2_relu_fp32, act_scales["relu2_out"])

    # FC3
    logits_fp32 = linear_int8(
        x_int8=fc2_relu_int8,
        w_int8=params["fc3"]["weight_int8"],
        x_scale=act_scales["relu2_out"],
        w_scale=params["fc3"]["weight_scale"],
        bias_fp32=params["fc3"]["bias_fp32"],
    )

    probs = softmax_fp32(logits_fp32)
    pred_class = int(np.argmax(probs, axis=1)[0])

    return pred_class, logits_fp32[0], probs[0]


def load_mnist_test():
    transform = transforms.Compose([
        transforms.ToTensor(),
    ])
    test_dataset = datasets.MNIST(
        root="./data",
        train=False,
        download=True,
        transform=transform
    )
    return test_dataset


def get_sample(test_dataset, idx: int):
    image, label = test_dataset[idx]
    image_np = image.numpy().astype(np.float32).reshape(-1)  # [784]
    return image_np, int(label)


def evaluate_manual_accuracy(
    test_dataset,
    params: Dict,
    act_scales: Dict,
    num_samples: int = 1000
) -> float:
    correct = 0
    total = min(num_samples, len(test_dataset))

    for i in range(total):
        image_np, label = get_sample(test_dataset, i)
        pred, _, _ = predict_one_manual(image_np, params, act_scales)
        if pred == label:
            correct += 1

    return correct / total


def benchmark_manual_inference(
    test_dataset,
    params: Dict,
    act_scales: Dict,
    num_samples: int = 1000
) -> float:
    total = min(num_samples, len(test_dataset))

    start = time.perf_counter()
    for i in range(total):
        image_np, _ = get_sample(test_dataset, i)
        _ = predict_one_manual(image_np, params, act_scales)
    end = time.perf_counter()

    avg_ms = (end - start) * 1000.0 / total
    return avg_ms



def main():
    print("Loading artifacts...")
    params, act_scales = load_artifacts()

    print("Loading MNIST test set...")
    test_dataset = load_mnist_test()


    print("\n===== Sample Predictions =====")
    for idx in [0, 1, 2, 3, 4]:
        image_np, label = get_sample(test_dataset, idx)
        pred, logits, probs = predict_one_manual(image_np, params, act_scales)

        print(f"Sample {idx}: label={label}, pred={pred}")
        print(f"  logits = {np.round(logits, 4)}")
        print(f"  probs  = {np.round(probs, 4)}")


    print("\n===== Manual Quantized Inference Accuracy =====")
    acc_1000 = evaluate_manual_accuracy(test_dataset, params, act_scales, num_samples=1000)
    print(f"Accuracy on first 1000 test samples: {acc_1000:.4f}")

    acc_full = evaluate_manual_accuracy(test_dataset, params, act_scales, num_samples=len(test_dataset))
    print(f"Accuracy on full test set: {acc_full:.4f}")

    print("\n===== Timing =====")
    avg_ms = benchmark_manual_inference(test_dataset, params, act_scales, num_samples=1000)
    print(f"Average manual inference latency: {avg_ms:.4f} ms/sample")


if __name__ == "__main__":
    main()

Loading artifacts...
Loading MNIST test set...

===== Sample Predictions =====
Sample 0: label=7, pred=7
  logits = [ -3.6928  -3.312    0.5122   0.4115  -6.4975  -4.1298 -13.5441   8.8216
  -2.2284  -0.8994]
  probs  = [0.000e+00 0.000e+00 2.000e-04 2.000e-04 0.000e+00 0.000e+00 0.000e+00
 9.994e-01 0.000e+00 1.000e-04]
Sample 1: label=2, pred=2
  logits = [ -4.9836   4.5175  11.2698   3.757  -12.6353  -2.426   -4.9066  -8.0107
  -0.9133 -14.982 ]
  probs  = [0.000e+00 1.200e-03 9.983e-01 5.000e-04 0.000e+00 0.000e+00 0.000e+00
 0.000e+00 0.000e+00 0.000e+00]
Sample 2: label=1, pred=1
  logits = [-4.1987  7.2566  0.4303 -2.8841 -1.5642 -4.0986 -2.7487  0.4315 -1.0785
 -4.0992]
  probs  = [0.000e+00 9.973e-01 1.100e-03 0.000e+00 1.000e-04 0.000e+00 0.000e+00
 1.100e-03 2.000e-04 0.000e+00]
Sample 3: label=0, pred=0
  logits = [ 9.7378 -4.9776 -1.7668 -2.879  -4.7552 -2.502  -0.6943 -2.1416 -7.6778
 -1.8443]
  probs  = [0.9999 0.     0.     0.     0.     0.     0.     0.     0.     0.  

In [ ]:
import numpy as np
import json
import torch
from pynq import Overlay, allocate
from torchvision import datasets, transforms

# Load quantized artifacts
qdict = torch.load("artifacts/mnist_mlp_int8_weights.pth", map_location="cpu")
with open("artifacts/activation_scales.json", "r") as f:
    act_scales = json.load(f)

w_fc1 = qdict["fc1.weight_int8"].numpy() # shape (128, 784), int8
b_fc1 = qdict["fc1.bias_fp32"].numpy() # shape (128,), float32
w_scale_fc1 = float(qdict["fc1.weight_scale"])

# Load overlay
FC1_M = 1
FC1_K = 784
FC1_N = 128

ol = Overlay("block_design.bit") # corret .bit name

ip = ol.multiply_0 # corrent IP core instance name

# Allocate DMA-capable buffers
A_buf = allocate(shape=(FC1_M * FC1_K,), dtype=np.int8)
B_buf = allocate(shape=(FC1_K * FC1_N,), dtype=np.int8)
C_buf = allocate(shape=(FC1_M * FC1_N,), dtype=np.int32)

# Load one MNIST sample
test_ds = datasets.MNIST(root="./data", train=False, download=True, transform=transforms.ToTensor())
image, label = test_ds[0]
x_fp32 = image.numpy().reshape(-1) # shape (784,)
x_int8 = np.clip(np.round(x_fp32 / act_scales["input"]), -128, 127).astype(np.int8)

# Fill buffers
A_buf[:] = x_int8 # shape (784,)
B_buf[:] = w_fc1.T.flatten() # weights row-major (128×784)

A_buf.sync_to_device()
B_buf.sync_to_device()

# Configure IP registers
ip.write(0x10, A_buf.device_address & 0xFFFFFFFF)
ip.write(0x14, (A_buf.device_address >> 32) & 0xFFFFFFFF)
ip.write(0x18, B_buf.device_address & 0xFFFFFFFF)
ip.write(0x1c, (B_buf.device_address >> 32) & 0xFFFFFFFF)
ip.write(0x20, C_buf.device_address & 0xFFFFFFFF)
ip.write(0x24, (C_buf.device_address >> 32) & 0xFFFFFFFF)
ip.write(0x28, FC1_M)
ip.write(0x30, FC1_K)
ip.write(0x38, FC1_N)

# Start and poll
ip.write(0x00, 0x01)

while (ip.read(0x00) & 0x2) == 0:
    pass

# Retrieve and dequantize
C_buf.sync_from_device()

y_int32 = np.array(C_buf).reshape(FC1_M, FC1_N) # (1, 128) int32
y_fp32  = y_int32.astype(np.float32) * (act_scales["input"] * w_scale_fc1)
y_fp32 += b_fc1 # add bias in float
y_relu  = np.maximum(y_fp32, 0.0) # ReLU then feed to FC2

print(f"Label: {label}, FC1 output (first 8): {y_relu[0, :8]}")

A_buf.close()
B_buf.close()
C_buf.close()